# 🔁 Reclaim — AI Revenue Recovery
### Razorpay Buildathon · Track 03

**Problem:** Failed payments silently kill merchant revenue. A ₹50,000 transaction that fails due to a UPI timeout isn't necessarily lost — the right recovery action at the right time can get it back.

**Reclaim** uses a trained ML model to recommend the optimal recovery action for each failed transaction, based on context: failure reason, customer risk tier, amount, and recovery history.

In [ ]:
import sys, os
os.chdir('..')  # run from project root so data/ paths work
sys.path.insert(0, '.')
import sys

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'text.color':       '#eee',
    'grid.color':       '#2a2a2a',
    'font.family':      'monospace',
})

print('✅ Imports ready')

## Step 1 — Train the model
Trains a Logistic Regression on 587 simulated failed transactions and saves the model.

In [ ]:
from src.train import train
model = train()

## Step 2 — A failed transaction arrives
Merchant receives a webhook: a ₹47,500 UPI payment just failed due to a timeout.

In [ ]:
failed_transaction = {
    'transaction_id':   'txn_demo_001',
    'amount':           47500,
    'payment_method':   'upi',
    'failure_reason':   'upi_timeout',
    'customer': {
        'risk_tier':    'medium',
        'prior_success_rate': 0.6,
    },
    'hours_since_failure': 1.5,
    'attempt_number':   1,
}

print('🚨 Failed transaction detected')
print(f"   Amount:          ₹{failed_transaction['amount']:,}")
print(f"   Method:          {failed_transaction['payment_method'].upper()}")
print(f"   Failure reason:  {failed_transaction['failure_reason']}")
print(f"   Customer risk:   {failed_transaction['customer']['risk_tier']}")

## Step 3 — Reclaim recommends the best action

In [ ]:
from src.agent import ReclaimAgent

agent = ReclaimAgent()
t = failed_transaction

results = agent.recommend(
    amount                    = t['amount'],
    payment_method            = t['payment_method'],
    failure_reason            = t['failure_reason'],
    risk_tier                 = t['customer']['risk_tier'],
    attempt_number            = t['attempt_number'],
    hours_since_failure       = t['hours_since_failure'],
    customer_prior_success_rate = t['customer']['prior_success_rate'],
)

agent.print_recommendation(results, amount=t['amount'])

## Step 4 — Visualise the recommendation

In [ ]:
labels = [r['label'] for r in results]
scores = [r['confidence'] * 100 for r in results]
colors = ['#4ade80' if i == 0 else '#4a9ade' for i in range(len(results))]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels[::-1], scores[::-1], color=colors[::-1], height=0.5)

for bar, score in zip(bars, scores[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{score:.1f}%', va='center', fontsize=11, color='#eee')

ax.set_xlim(0, 100)
ax.set_xlabel('Predicted success probability (%)')
ax.set_title(f'Reclaim — Recovery Action Ranking  |  ₹{t["amount"]:,} at risk', pad=12, fontsize=13)
ax.axvline(50, color='#555', linestyle='--', linewidth=0.8, label='50% baseline')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

best_patch = mpatches.Patch(color='#4ade80', label='Recommended action')
other_patch = mpatches.Patch(color='#4a9ade', label='Other options')
ax.legend(handles=[best_patch, other_patch], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('data/recommendation_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Revenue Recovery Comparison
Reclaim vs. the naive baseline: *always retry after delay*

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/ml_dataset.csv')
X = df.drop(columns=['recovery_success'])
y = df['recovery_success']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_pred = model.predict(X_test)
amounts = X_test['amount'].values
actual  = y_test.values

total_at_risk      = amounts.sum()
actual_recoverable = amounts[actual == 1].sum()
reclaim_recovered  = amounts[y_pred == 1].sum()

baseline_mask     = X_test['action'].values == 1   # retry_after_delay
baseline_coverage = baseline_mask.mean()
baseline_recovered = (amounts[baseline_mask & (actual == 1)].sum() / baseline_coverage
                      if baseline_coverage > 0 else 0)

categories  = ['Max\nRecoverable', 'Reclaim\n(ML)', 'Baseline\n(retry later)']
values      = [actual_recoverable, reclaim_recovered, baseline_recovered]
bar_colors  = ['#a78bfa', '#4ade80', '#f87171']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(categories, [v / 1e5 for v in values], color=bar_colors, width=0.4)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'₹{val:,.0f}\n({val/total_at_risk*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, color='#eee')

ax.set_ylabel('Revenue recovered (₹ lakhs)')
ax.set_title('Revenue Recovery: Reclaim vs Baseline', pad=12, fontsize=13)
ax.set_ylim(0, max(v / 1e5 for v in values) * 1.35)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/revenue_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

uplift = (reclaim_recovered - baseline_recovered) / baseline_recovered * 100
print(f'\nReclaim recovers {uplift:.0f}% more revenue than the baseline strategy.')

## How it works

```
Failed Transaction
      │
      ▼
Feature Builder ──► amount, payment_method, failure_reason,
                    risk_tier, attempt_number, hours_since_failure,
                    customer_prior_success_rate
      │
      ▼
Reclaim Model ──► scores all 4 recovery actions
      │
      ▼
Recommendation ──► best action + confidence
      │
      ▼
Merchant takes action ──► outcome logged ──► model improves
```

**Model:** Logistic Regression with StandardScaler  
**Training data:** 587 simulated transactions with context-aware outcomes  
**Key insight:** The same failed transaction needs different recovery depending on *why* it failed and *who* the customer is.